# **Insatll dependencies & Imports**

In [ ]:
!pip install langchain langchain-openai tavily-python requests

In [ ]:
import os
import requests

from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain.agents import create_agent
from tavily import TavilyClient

# **Set Environment Variables**

In [ ]:
try:
    # In Colab? read from userdata (secrets)
    from google.colab import userdata
    ON_COLAB = True
    os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
    os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")
except ImportError:
    pass
    from google.colab import userdata


# **Create the Internet Search Tool & the fetch_url Tool**

In [ ]:
tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

@tool
def internet_search(query: str):
    """Search the internet and return the top 3 results."""
    return tavily_client.search(query, max_results=3)

In [ ]:
@tool
def fetch_url(url: str) -> str:
    """Fetch text content from a URL"""
    response = requests.get(url, timeout=10.0)
    response.raise_for_status()
    return response.text

# **Initialize the LLM (OpenRouter Model)**

In [ ]:
# https://openrouter.ai/nvidia/nemotron-3-nano-30b-a3b:free
model_nemotron3_nano = ChatOpenAI(
    model="nvidia/nemotron-3-nano-30b-a3b:free",
    temperature=0,
    # OpenRouter instead of the default OpenAI API
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY"),
)

# **System Prompt**

In [ ]:
system_prompt = """
You are an AI research assistant.

Your job is to:
1. Search the internet for the user's question.
2. Use the top 3 search results.
3. Fetch the content from those websites.
4. Combine the information and give a clear answer.

Always use multiple sources when answering.
"""

# **Create the Agent**

In [ ]:
agent = create_agent(
    model=model_nemotron3_nano,
    tools=[internet_search, fetch_url],
    system_prompt=system_prompt
)

In [ ]:
print([internet_search, fetch_url])

[StructuredTool(name='internet_search', description='Search the internet and return the top 3 results.', args_schema=<class 'langchain_core.utils.pydantic.internet_search'>, func=<function internet_search at 0x7d8084a33880>), StructuredTool(name='fetch_url', description='Fetch text content from a URL', args_schema=<class 'langchain_core.utils.pydantic.fetch_url'>, func=<function fetch_url at 0x7d8063d3ef20>)]


# **Testing the agent**

In [ ]:
result = agent.invoke({
    "messages": [
        {"role": "user", "content": "What is agentic AI?"}
    ]
})

print(result["messages"][-1].content)

**Agentic AI** is an autonomous artificial‑intelligence system that can **plan, decide, and take actions on its own** to achieve a specific goal. Unlike traditional AI that only generates output (e.g., text, images), an agentic system:

* **Acts autonomously** – it can execute multi‑step tasks without constant human prompts.  
* **Reasons through problems** – it breaks complex objectives into smaller steps, evaluates options, and chooses the best course of action.  
* **Adapts in real‑time** – it continuously monitors its environment, learns from new data, and adjusts its behavior to stay on target.  
* **Collaborates** – multiple agents can work together, sharing information and dividing labor to tackle larger challenges.  

### Key Characteristics (from the top sources)

| Source | Core Idea |
|--------|-----------|
| **Salesforce** – *What is Agentic AI?* | An intelligent system that can act autonomously, reason through multi‑step problems, and adapt its actions in real‑time to achi